In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Transpilação com gerenciadores de passagem

<details>
<summary><b>Package versions</b></summary>

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</details>
A forma recomendada de transpilar um circuito é criar um gerenciador de passagem em estágios e, em seguida, executar o método `run` com o circuito como entrada. Esta página explica como transpilar circuitos quânticos dessa maneira.
## O que é um gerenciador de passagem (em estágios)?
No contexto do Qiskit SDK, transpilação refere-se ao processo de transformar um circuito de entrada em uma forma adequada para execução em um dispositivo quântico. A transpilação geralmente ocorre em uma sequência de etapas chamadas passes de transpilação. O circuito é processado por cada passe de transpilação em sequência, sendo que a saída de um passe se torna a entrada do próximo. Por exemplo, um passe pode percorrer o circuito e mesclar todas as sequências consecutivas de portas de um único qubit, e então o próximo passe pode sintetizar essas portas no conjunto de base do dispositivo alvo. Os passes de transpilação incluídos no Qiskit estão localizados no módulo [qiskit.transpiler.passes](https://docs.quantum.ibm.com/api/qiskit/transpiler_passes).

Um gerenciador de passagem é um objeto que armazena uma lista de passes de transpilação e pode executá-los em um circuito. Crie um gerenciador de passagem inicializando um [`PassManager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager) com uma lista de passes de transpilação. Para executar a transpilação em um circuito, chame o método [`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager#run) com um circuito como entrada.

Um gerenciador de passagem em estágios é um tipo especial de gerenciador de passagem que representa um nível de abstração acima do gerenciador de passagem normal. Enquanto um gerenciador de passagem normal é composto por vários passes de transpilação, um gerenciador de passagem em estágios é composto por vários *gerenciadores de passagem*. Essa é uma abstração útil porque a transpilação geralmente ocorre em estágios discretos, conforme descrito em [Estágios do transpilador](transpiler-stages), com cada estágio sendo representado por um gerenciador de passagem. Os gerenciadores de passagem em estágios são representados pela classe [`StagedPassManager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.StagedPassManager). O restante desta página descreve como criar e personalizar gerenciadores de passagem (em estágios).
## Gerar um gerenciador de passagem em estágios predefinido
Para criar um gerenciador de passagem em estágios predefinido com padrões razoáveis, use a função [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager):

In [1]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()
pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

Para transpilar um circuito ou uma lista de circuitos com um gerenciador de passagem, passe o circuito ou a lista de circuitos para o método `run`. Vamos fazer isso em um circuito de dois qubits composto por uma porta Hadamard seguida de duas portas CX adjacentes:

In [2]:
from qiskit import QuantumRegister, QuantumCircuit

# Create a circuit
qubits = QuantumRegister(2, name="q")
circuit = QuantumCircuit(qubits)
a, b = qubits
circuit.h(a)
circuit.cx(a, b)
circuit.cx(b, a)

# Transpile it by calling the run method of the pass manager
transpiled = pass_manager.run(circuit)

# Draw it, excluding idle qubits from the diagram
transpiled.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/transpile-with-pass-managers/extracted-outputs/c1426e6c-f506-4938-8c0a-05198bae9746-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpile-with-pass-managers/extracted-outputs/dcc69b72-e13b-4df6-a51f-a5ef2108bae7-0.svg)

Consulte [Configurações padrão de transpilação e opções de configuração](defaults-and-configuration-options) para uma descrição dos possíveis argumentos da função `generate_preset_pass_manager`. Os argumentos de `generate_preset_pass_manager` correspondem aos argumentos da função [`transpile`](https://docs.quantum.ibm.com/api/qiskit/compiler#qiskit.compiler.transpile).

<CodeAssistantAdmonition tagLine="Having trouble remembering pass manager details? Try asking Qiskit Code Assistant." />

Se os gerenciadores de passagem predefinidos não atenderem às suas necessidades, personalize a transpilação criando gerenciadores de passagem (em estágios) ou até mesmo passes de transpilação. O restante desta página descreve como criar gerenciadores de passagem. Para instruções sobre como criar passes de transpilação, consulte [Escreva seu próprio passe de transpilador](custom-transpiler-pass).

## Crie seu próprio gerenciador de passagem

O módulo [qiskit.transpiler.passes](https://docs.quantum.ibm.com/api/qiskit/transpiler_passes) inclui muitos passes de transpilação que podem ser usados para criar gerenciadores de passagem. Para criar um gerenciador de passagem, inicialize um `PassManager` com uma lista de passes. Por exemplo, o código a seguir cria um passe de transpilação que mescla portas adjacentes de dois qubits e, em seguida, as sintetiza em uma base de portas [$R_y$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RYGate), [$R_z$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RZGate) e [$R_{xx}$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RXXGate).

In [3]:
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import (
    Collect2qBlocks,
    ConsolidateBlocks,
    UnitarySynthesis,
)

basis_gates = ["rx", "ry", "rxx"]
translate = PassManager(
    [
        Collect2qBlocks(),
        ConsolidateBlocks(basis_gates=basis_gates),
        UnitarySynthesis(basis_gates),
    ]
)

Para demonstrar esse gerenciador de passagem em ação, teste-o em um circuito de dois qubits composto por uma porta Hadamard seguida de duas portas CX adjacentes:

In [4]:
from qiskit import QuantumRegister, QuantumCircuit

qubits = QuantumRegister(2, name="q")
circuit = QuantumCircuit(qubits)

a, b = qubits
circuit.h(a)
circuit.cx(a, b)
circuit.cx(b, a)

circuit.draw("mpl")

<Image src="../docs/images/guides/transpile-with-pass-managers/extracted-outputs/01317c54-68b5-4e41-893f-82ee223e22f0-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpile-with-pass-managers/extracted-outputs/bc208935-e63c-461b-90d0-a6f4cabc16b6-0.svg)

Para executar o gerenciador de passagem no circuito, chame o método `run`.

In [5]:
translated = translate.run(circuit)
translated.draw("mpl")

<Image src="../docs/images/guides/transpile-with-pass-managers/extracted-outputs/019ad99b-bd38-4217-90ee-da43959dc8ad-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpile-with-pass-managers/extracted-outputs/adb5c242-5cba-4878-a00d-4ec47737d029-0.svg)

Para um exemplo mais avançado que mostra como criar um gerenciador de passagem para implementar a técnica de supressão de erros conhecida como desacoplamento dinâmico, consulte [Criar um gerenciador de passagem para desacoplamento dinâmico](dynamical-decoupling-pass-manager).

## Criar um gerenciador de passagem em estágios

Um `StagedPassManager` é um gerenciador de passagem composto por estágios individuais, onde cada estágio é definido por uma instância de `PassManager`. Você pode criar um `StagedPassManager` especificando os estágios desejados. Por exemplo, o código a seguir cria um gerenciador de passagem em estágios com dois estágios, `init` e `translation`. O estágio `translation` é definido pelo gerenciador de passagem criado anteriormente.

In [6]:
from qiskit.transpiler import PassManager, StagedPassManager
from qiskit.transpiler.passes import UnitarySynthesis, Unroll3qOrMore

basis_gates = ["rx", "ry", "rxx"]
init = PassManager(
    [UnitarySynthesis(basis_gates, min_qubits=3), Unroll3qOrMore()]
)
staged_pm = StagedPassManager(
    stages=["init", "translation"], init=init, translation=translate
)

Não há limite para o número de estágios que você pode incluir em um gerenciador de passagem em estágios.

Outra forma útil de criar um gerenciador de passagem em estágios é começar com um gerenciador de passagem em estágios predefinido e substituir alguns dos estágios. Por exemplo, o código a seguir gera um gerenciador de passagem predefinido com nível de otimização 3 e, em seguida, especifica um estágio `pre_layout` personalizado.

In [7]:
import numpy as np
from qiskit.circuit.library import HGate, PhaseGate, RXGate, TdgGate, TGate
from qiskit.transpiler.passes import InverseCancellation

pass_manager = generate_preset_pass_manager(3, backend)
inverse_gate_list = [
    HGate(),
    (RXGate(np.pi / 4), RXGate(-np.pi / 4)),
    (PhaseGate(np.pi / 4), PhaseGate(-np.pi / 4)),
    (TGate(), TdgGate()),
]
logical_opt = PassManager(
    [
        InverseCancellation(inverse_gate_list),
    ]
)

# Add pre-layout stage to run extra logical optimization
pass_manager.pre_layout = logical_opt

As [funções geradoras de estágios](https://docs.quantum.ibm.com/api/qiskit/transpiler_preset#stage-generator-functions) podem ser úteis para construir gerenciadores de passagem personalizados.
Elas geram estágios que oferecem funcionalidades comuns usadas em muitos gerenciadores de passagem.
Por exemplo, [`generate_embed_passmanager`](https://docs.quantum.ibm.com/api/qiskit/transpiler_preset#qiskit.transpiler.preset_passmanagers.generate_embed_passmanager) pode ser usado para gerar um estágio
para "incorporar" um `Layout` inicial selecionado de um passe de layout no dispositivo alvo especificado.

## Próximas etapas
> **Tip:** - [Escrever um passe de transpilador personalizado](custom-transpiler-pass).
>     - [Criar um gerenciador de passagem para desacoplamento dinâmico](dynamical-decoupling-pass-manager).
>     - Para saber mais sobre a função `generate_preset_passmanager`, consulte o tópico [Configurações padrão e opções de configuração do transpilador](defaults-and-configuration-options).
>     - Experimente o guia [Comparar configurações do transpilador](/guides/circuit-transpilation-settings).
>     - Revise a [documentação da API do transpilador.](https://docs.quantum.ibm.com/api/qiskit/transpiler)